# RAG Pipeline with FAISS HNSW, Cross-Encoder Reranking, and Evaluation

**Based on:** `RAG_using_FaissHNSWIndex_BERTpaper.ipynb`. This revised notebook keeps the original PDF → chunking → embeddings → FAISS HNSW → reranking → generation workflow, while correcting key issues and adding reproducible evaluation cells.

**Important:** evaluation scores are not pre-filled or claimed. Run the notebook, then supply human-judged relevance labels for the evaluation questions. Automatic keyword checks are only a rough diagnostic, not a substitute for relevance judgments.

Recommended runtime: Google Colab (CPU works, GPU can speed up models).

## 1. Install dependencies

In [1]:
%pip install -q pymupdf faiss-cpu sentence-transformers transformers accelerate pandas numpy matplotlib scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 47.5 MB/s eta 0:00:00


In [2]:
import json
import os
import random
import re
import time
from pathlib import Path

import faiss
import fitz  # PyMuPDF
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sentence_transformers import CrossEncoder, SentenceTransformer
from transformers import AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "| FAISS:", faiss.__version__)

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Device: cuda | FAISS: 1.15.1


## 2. Ingest PDF

In [3]:
# Default source from the original notebook: BERT paper on arXiv.
PDF_URL = "https://arxiv.org/pdf/1810.04805"
PDF_PATH = Path("/content/data.pdf" if Path("/content").exists() else "data.pdf")

# If you have your own PDF, set PDF_PATH to its path and set DOWNLOAD_PDF=False.
DOWNLOAD_PDF = True
if DOWNLOAD_PDF:
    import urllib.request

    urllib.request.urlretrieve(PDF_URL, PDF_PATH)

if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"PDF not found: {PDF_PATH}. Upload a PDF or update PDF_PATH."
    )

with fitz.open(PDF_PATH) as pdf:
    pages = [
        {"page": i + 1, "text": page.get_text("text")} for i, page in enumerate(pdf)
    ]

print(
    f'Loaded {PDF_PATH} | pages={len(pages)} | chars={sum(len(p["text"]) for p in pages):,}'
)
print(pages[0]["text"][:500])

Loaded /content/data.pdf | pages=16 | chars=64,117
BERT: Pre-training of Deep Bidirectional Transformers for
Language Understanding
Jacob Devlin
Ming-Wei Chang
Kenton Lee
Kristina Toutanova
Google AI Language
{jacobdevlin,mingweichang,kentonl,kristout}@google.com
Abstract
We introduce a new language representa-
tion model called BERT, which stands for
Bidirectional Encoder Representations from
Transformers. Unlike recent language repre-
sentation models (Peters et al., 2018a; Rad-
ford et al., 2018), BERT is designed to pre-
train deep bidirecti


## 3. Clean text and create overlapping chunks

In [4]:
def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"-\s*\n\s*", "", text)  # join line-break hyphenation
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def chunk_text(text: str, chunk_words: int = 180, overlap_words: int = 40):
    words = text.split()
    if not words:
        return []
    chunks = []
    step = max(1, chunk_words - overlap_words)
    for start in range(0, len(words), step):
        part = words[start : start + chunk_words]
        if part:
            chunks.append(" ".join(part))
        if start + chunk_words >= len(words):
            break
    return chunks


chunks = []
for page in pages:
    page_text = clean_text(page["text"])
    for local_idx, chunk in enumerate(chunk_text(page_text)):
        chunks.append({"chunk_id": len(chunks), "page": page["page"], "text": chunk})

if not chunks:
    raise ValueError(
        "No text chunks were extracted. Check whether the PDF is scanned/image-only."
    )
chunk_df = pd.DataFrame(chunks)
print(
    "Chunks:",
    len(chunk_df),
    "| Mean words:",
    round(chunk_df.text.str.split().str.len().mean(), 1),
)
display(chunk_df.head(3))

Chunks: 73 | Mean words: 165.3


,chunk_id,page,text
0,0,1,BERT: Pre-training of Deep Bidirectional Trans...
1,1,1,"absolute improvement), MultiNLI accuracy to 86..."
2,2,1,existing strategies for applying pre-trained l...


## 4. Embed chunks correctly

The original notebook passed token IDs to FAISS as if they were semantic embeddings. Token IDs are not dense semantic vectors. This version uses `SentenceTransformer.encode()` to create real sentence embeddings and normalizes them for cosine-similarity search.

In [5]:
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBED_MODEL_NAME, device=device)
corpus_texts = chunk_df["text"].tolist()

t0 = time.perf_counter()
corpus_embeddings = embedding_model.encode(
    corpus_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")
embedding_seconds = time.perf_counter() - t0
print(
    "Embedding matrix:",
    corpus_embeddings.shape,
    "| seconds:",
    round(embedding_seconds, 2),
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embedding matrix: (73, 384) | seconds: 2.44


## 5. Build FAISS HNSW index

In [6]:
embedding_dim = corpus_embeddings.shape[1]
hnsw = faiss.IndexHNSWFlat(embedding_dim, 32, faiss.METRIC_INNER_PRODUCT)
hnsw.hnsw.efConstruction = 80
hnsw.hnsw.efSearch = 64
hnsw.add(corpus_embeddings)
print("Indexed vectors:", hnsw.ntotal, "| trained:", hnsw.is_trained)

Indexed vectors: 73 | trained: True


## 6. Retrieval function and quick smoke test

In [7]:
def retrieve(query: str, k: int = 10):
    query_embedding = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")
    k = min(max(1, int(k)), hnsw.ntotal)
    start = time.perf_counter()
    scores, ids = hnsw.search(query_embedding, k)
    elapsed_ms = (time.perf_counter() - start) * 1000
    results = []
    for rank, (idx, score) in enumerate(zip(ids[0], scores[0]), start=1):
        if idx < 0:
            continue
        row = chunk_df.iloc[int(idx)]
        results.append(
            {
                "rank": rank,
                "chunk_id": int(row.chunk_id),
                "page": int(row.page),
                "score": float(score),
                "text": row.text,
            }
        )
    return results, elapsed_ms


query = "How does BERT use bidirectional context to learn language representations?"
smoke_results, smoke_latency = retrieve(query, k=5)
print(f"Retrieval latency: {smoke_latency:.2f} ms")
for item in smoke_results:
    print(
        f"\nRank {item['rank']} | page {item['page']} | score {item['score']:.3f}\n{item['text'][:450]}..."
    )

Retrieval latency: 0.14 ms

Rank 1 | page 4 | score 0.699
with a special token ([SEP]). Second, we add a learned embedding to every token indicating whether it belongs to sentence A or sentence B. As shown in Figure 1, we denote input embedding as E, the ﬁnal hidden vector of the special [CLS] token as C ∈RH, and the ﬁnal hidden vector for the ith input token as Ti ∈RH. For a given token, its input representation is constructed by summing the corresponding token, segment, and position embeddings. A visu...

Rank 2 | page 1 | score 0.691
BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding Jacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova Google AI Language {jacobdevlin,mingweichang,kentonl,kristout}@google.com Abstract We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Transformers. Unlike recent language representation models (Peters et al., 2018a; Radford et al., 2018), B...

Ra

## 7. Optional cross-encoder reranking

The original notebook used `BertForQuestionAnswering` start/end logits as a reranking score. Those logits are trained for answer-span prediction and are not a well-calibrated relevance score. This version uses a cross-encoder trained for passage relevance. Model downloads may take a few minutes.

In [8]:
RERANK_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANK_MODEL_NAME, device=device)


def rerank(query: str, candidates: list, top_n: int = 5):
    if not candidates:
        return []
    pairs = [(query, item["text"]) for item in candidates]
    scores = reranker.predict(pairs, show_progress_bar=False)
    ranked = []
    for item, score in zip(candidates, scores):
        copy = dict(item)
        copy["rerank_score"] = float(score)
        ranked.append(copy)
    return sorted(ranked, key=lambda x: x["rerank_score"], reverse=True)[:top_n]


# Example: rerank top 10 retrieved passages
initial_results, _ = retrieve(query, k=10)
reranked_results = rerank(query, initial_results, top_n=5)
for rank, item in enumerate(reranked_results, start=1):
    print(
        f"\nRank {rank} | page {item['page']} | rerank score {item['rerank_score']:.3f}\n{item['text'][:450]}..."
    )

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


Rank 1 | page 2 | rerank score 6.186
word based only on its context. Unlike left-toright language model pre-training, the MLM objective enables the representation to fuse the left and the right context, which allows us to pretrain a deep bidirectional Transformer. In addition to the masked language model, we also use a “next sentence prediction” task that jointly pretrains text-pair representations. The contributions of our paper are as follows: • We demonstrate the importance of bi...

Rank 2 | page 1 | rerank score 5.626
BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding Jacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova Google AI Language {jacobdevlin,mingweichang,kentonl,kristout}@google.com Abstract We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Transformers. Unlike recent language representation models (Peters et al., 2018a; Radford et al., 2018), B...

Rank 3 | page 1

## 8. Retrieval evaluation with labelled relevance judgments

Add a small set of evaluation questions and manually label the chunk IDs that are relevant for each question. Use the retrieved results below to find candidate IDs, then update `RELEVANCE_JUDGMENTS`. Do not use guessed or generated labels as ground truth.

In [9]:
# Start with questions about the source paper. Edit these after inspecting the document.
EVAL_QUESTIONS = [
    "What is the main idea behind BERT?",
    "How does BERT differ from left-to-right language models?",
    "What pre-training tasks are used by BERT?",
    "How is BERT fine-tuned for downstream tasks?",
    "What datasets or benchmarks are used to evaluate BERT?",
]

# TODO: fill each list with chunk IDs that a human reviewer judges relevant.
# Example format only: {'question': [3, 8]} -- do not use example IDs without checking them.
RELEVANCE_JUDGMENTS = {q: [] for q in EVAL_QUESTIONS}

# Display candidate passages and IDs to make labelling easier.
for q in EVAL_QUESTIONS:
    results, _ = retrieve(q, k=5)
    print("\nQUESTION:", q)
    for r in results:
        print(
            f"  chunk_id={r['chunk_id']} page={r['page']} score={r['score']:.3f} | {r['text'][:220]}..."
        )


QUESTION: What is the main idea behind BERT?
  chunk_id=14 page=4 score=0.489 | with a special token ([SEP]). Second, we add a learned embedding to every token indicating whether it belongs to sentence A or sentence B. As shown in Figure 1, we denote input embedding as E, the ﬁnal hidden vector of t...
  chunk_id=11 page=3 score=0.479 | ﬁnetuning, the BERT model is ﬁrst initialized with the pre-trained parameters, and all of the parameters are ﬁne-tuned using labeled data from the downstream tasks. Each downstream task has separate ﬁne-tuned models, eve...
  chunk_id=13 page=4 score=0.467 | Input/Output Representations To make BERT handle a variety of down-stream tasks, our input representation is able to unambiguously represent both a single sentence and a pair of sentences (e.g., ⟨Question, Answer ⟩) in o...
  chunk_id=56 page=13 score=0.440 | BERT (Ours) Trm Trm Trm Trm Trm Trm ... ... Trm Trm Trm Trm Trm Trm ... ... OpenAI GPT Lstm ELMo Lstm Lstm Lstm Lstm Lstm Lstm Lstm Lstm Lstm

In [10]:
def dcg(relevances):
    return sum((2**rel - 1) / np.log2(i + 2) for i, rel in enumerate(relevances))


def retrieval_metrics(retrieved_ids, relevant_ids, k=5):
    relevant_ids = set(relevant_ids)
    top_ids = list(retrieved_ids[:k])
    hits = [1 if cid in relevant_ids else 0 for cid in top_ids]
    recall = (
        len(set(top_ids) & relevant_ids) / len(relevant_ids) if relevant_ids else np.nan
    )
    precision = sum(hits) / k
    rr = next((1.0 / rank for rank, hit in enumerate(hits, 1) if hit), 0.0)
    # Binary graded relevance for nDCG
    ideal = sorted(hits, reverse=True)
    idcg = dcg(ideal)
    ndcg = dcg(hits) / idcg if idcg else np.nan
    return {"Recall@k": recall, "Precision@k": precision, "MRR@k": rr, "nDCG@k": ndcg}


def evaluate_retrieval(questions, judgments, k=5, use_reranker=False):
    rows = []
    for q in questions:
        relevant = judgments.get(q, [])
        if not relevant:
            continue
        candidates, latency_ms = retrieve(q, k=max(k, 10) if use_reranker else k)
        if use_reranker:
            candidates = rerank(q, candidates, top_n=k)
        ids = [x["chunk_id"] for x in candidates[:k]]
        row = {
            "question": q,
            "latency_ms": latency_ms,
            "labelled_relevant_chunks": len(relevant),
        }
        row.update(retrieval_metrics(ids, relevant, k=k))
        rows.append(row)
    return pd.DataFrame(rows)


# Run after filling RELEVANCE_JUDGMENTS above.
baseline_eval = evaluate_retrieval(
    EVAL_QUESTIONS, RELEVANCE_JUDGMENTS, k=5, use_reranker=False
)
reranked_eval = evaluate_retrieval(
    EVAL_QUESTIONS, RELEVANCE_JUDGMENTS, k=5, use_reranker=True
)
if baseline_eval.empty:
    print(
        "No retrieval metrics calculated yet. Fill RELEVANCE_JUDGMENTS with human-reviewed chunk IDs, then rerun this cell."
    )
else:
    display(baseline_eval)
    display(reranked_eval)
    summary = pd.DataFrame(
        [
            {
                "pipeline": "Dense retrieval",
                **baseline_eval.select_dtypes("number").mean().to_dict(),
            },
            {
                "pipeline": "Dense retrieval + reranking",
                **reranked_eval.select_dtypes("number").mean().to_dict(),
            },
        ]
    )
    display(summary)

No retrieval metrics calculated yet. Fill RELEVANCE_JUDGMENTS with human-reviewed chunk IDs, then rerun this cell.


## 9. Generate an answer from retrieved context (optional)

GPT-2 is a base language model, not an instruction-tuned QA model. Treat its output as a prototype demonstration, not proof of answer correctness. For a stronger QA system, replace it with a suitable instruction-tuned model that your hardware and license allow.

In [11]:
GEN_MODEL_NAME = "google/flan-t5-small"
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL_NAME).to(device)


def answer_question(question: str, contexts: list, max_new_tokens: int = 160):
    context_text = "\n\n".join(f"[Page {c['page']}] {c['text']}" for c in contexts[:4])
    prompt = (
        "Answer the question using only the context. If the context does not contain the answer, say you do not know.\n\n"
        f"Context:\n{context_text}\n\nQuestion: {question}\nAnswer:"
    )
    inputs = gen_tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=512
    ).to(device)
    with torch.inference_mode():
        output = gen_model.generate(
            **inputs, max_new_tokens=max_new_tokens, num_beams=2
        )
    return gen_tokenizer.decode(output[0], skip_special_tokens=True)


question = EVAL_QUESTIONS[0]
candidates, _ = retrieve(question, k=10)
context_for_answer = rerank(question, candidates, top_n=4)
answer = answer_question(question, context_for_answer)
print("Question:", question, "\nAnswer:", answer)
print("\nSource pages:", sorted(set(x["page"] for x in context_for_answer)))

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Question: What is the main idea behind BERT? 
Answer: [Page 3] finetuning, the BERT model is first initialized with the pre-trained parameters, and all of the parameters are fine-tuned using labeled data from the downstream tasks. Each downstream task has separate fine-tuned models, even though they are initialized with the same pre-trained parameters.

Source pages: [3, 4, 5, 14]


## 10. End-to-end evaluation template

For each evaluation question, record a reference answer, retrieved relevant chunk IDs, generated answer, human correctness rating, groundedness rating, and latency. These are manual evaluation fields; this notebook does not fabricate answer-quality scores.

In [12]:
# Edit/extend with your own reviewed test cases after running the pipeline.
qa_test_cases = pd.DataFrame(
    [
        {
            "question": q,
            "reference_answer": "",
            "relevant_chunk_ids": "",
            "generated_answer": "",
            "correctness_1_to_5": np.nan,
            "groundedness_1_to_5": np.nan,
            "notes": "",
        }
        for q in EVAL_QUESTIONS
    ]
)
display(qa_test_cases)
# After manual review, export the completed table:
# qa_test_cases.to_csv('rag_qa_evaluation.csv', index=False)

,question,reference_answer,relevant_chunk_ids,generated_answer,correctness_1_to_5,groundedness_1_to_5,notes
0,What is the main idea behind BERT?,,,,NaN,NaN,
1,How does BERT differ from left-to-right langua...,,,,NaN,NaN,
2,What pre-training tasks are used by BERT?,,,,NaN,NaN,
3,How is BERT fine-tuned for downstream tasks?,,,,NaN,NaN,
4,What datasets or benchmarks are used to evalua...,,,,NaN,NaN,


## 11. Latency and corpus summary

In [13]:
corpus_summary = pd.DataFrame(
    [
        {
            "pdf_file": str(PDF_PATH),
            "pages": len(pages),
            "chunks": len(chunk_df),
            "embedding_dimension": int(corpus_embeddings.shape[1]),
            "embedding_model": EMBED_MODEL_NAME,
            "embedding_seconds": round(embedding_seconds, 2),
            "index_vectors": int(hnsw.ntotal),
            "index_type": "FAISS IndexHNSWFlat (inner product on normalized vectors)",
            "device": device,
        }
    ]
)
display(corpus_summary)

,pdf_file,pages,chunks,embedding_dimension,embedding_model,embedding_seconds,index_vectors,index_type,device
0,/content/data.pdf,16,73,384,sentence-transformers/all-MiniLM-L6-v2,2.44,73,FAISS IndexHNSWFlat (inner product on normaliz...,cuda


## 12. Reporting results responsibly

Before reporting metrics on a resume or portfolio:
- Use a fixed, documented set of evaluation questions.
- Have a human inspect and label relevant chunks before computing Recall@k, MRR, and nDCG.
- Report baseline retrieval and reranked retrieval on the same questions.
- Measure end-to-end latency separately from retrieval latency.
- Manually score generated answer correctness and groundedness against source passages.
- Record model names, package versions, random seed, PDF source, chunk settings, and runtime hardware.
- Do not describe heuristic keyword matches or unreviewed model judgments as validated evaluation.

**Resume-ready output:** fill the labelled evaluation set, run all cells, save the metrics tables, and only then add measured retrieval or QA numbers to your resume.